Notebook to test image processing. 

In [1]:
import cv2
import numpy as np
from skimage import measure, morphology
import matplotlib.pyplot as plt
import os
import glob
import itertools
import csv
import pandas as pd

# functions

In [2]:
def collect_image_files(directory, extension='*.jpg'):
    # Search for image files in the given directory and all its subdirectories
    search_pattern = os.path.join(directory, '**', extension)
    image_files = glob.glob(search_pattern, recursive=True)
    return image_files




def analyze_hyphae_cells(image_path, threshold_value, min_size=30, pix=255, min_area=10, max_area=200, ecc_threshold=0.95, aspect_threshold=4,solidity_threshold=0.5, min_length=30):
    '''
    function to process microscopy images to determine objects that are hyphae.
    the image processing puts the image through
    1. grayscale
    2. thresholding
    3. small object removal
    4. finds props in photo
    5. draws contours around the hyphae. 

    image_path - string to image file. 
    threshold_value - int of the threshold
    min_size - small object removal threshold
    pix - pixels to use in the processed image
    min_area - minimal area to count as hyphae
    max_area = max area to count as hyphae
    ecc_threshold - eccentric shape -- 0 = sphere, 1 = non-spherical
    aspect_threshold - len/width threshold.
    min_length = min length to be hypahe cell.

    
    '''
    
    
    
    # Load the image
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError("Error: Image not found at given path.")
    # print(f"\n\n\n Analyzing {image_path} with params: {threshold_value}, {min_size}, {pix}, {min_area}, {max_area}")
    print(f"\nAnalyzing {image_path} with the following parameters:")
    print(f"  threshold_value: {threshold_value}")
    print(f"  min_size: {min_size}")
    print(f"  pix: {pix}")
    print(f"  min_area: {min_area}")
    print(f"  max_area: {max_area}")
    print(f"  ecc_threshold: {ecc_threshold}")
    print(f"  aspect_threshold: {aspect_threshold}")
    print(f"  solidity_threshold: {solidity_threshold}")
    print(f"  min_length: {min_length}")
    # Apply a threshold to segment the cells
    _, thresholded = cv2.threshold(image, threshold_value, pix, cv2.THRESH_BINARY_INV)
    
    # Remove small objects (noise)
    cleaned = morphology.remove_small_objects(thresholded.astype(bool), min_size=min_size).astype(np.uint8)

    # Label the connected components
    labels = measure.label(cleaned)
    props = measure.regionprops(labels)
    
    total_cells = 0
    hyphae_cells = 0
    hyphae_areas = []
    hyphae_aspect_ratios = []
    
    # Create a copy of the original image to draw annotations
    annotated_image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)

    for prop in props:
        if min_area < prop.area:  # Threshold area to count as a cell, adjust as needed
            total_cells += 1

            mask = np.zeros_like(labels, dtype=np.uint8)
            mask[labels == prop.label] = 255

            contours, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_NONE)
            
            if len(contours) == 0:
                continue
            
            contour = contours[0]
            
            aspect_ratio = prop.major_axis_length / prop.minor_axis_length if prop.minor_axis_length != 0 else 0
            solidity = prop.area / prop.convex_area if prop.convex_area != 0 else 0
            major_length = prop.major_axis_length

            
            if (prop.eccentricity > ecc_threshold) and (aspect_ratio > aspect_threshold) and (solidity < solidity_threshold) and (major_length > min_length):
                hyphae_cells += 1
                hyphae_areas.append(prop.area)
                hyphae_aspect_ratios.append(aspect_ratio)
                # cv2.drawContours(annotated_image, [contour], 0, (0, 0, 139), 2)
                cv2.drawContours(annotated_image, [contour], 0, (57, 255, 20), 2)
                
                label_text = f'Hyphae AR:{aspect_ratio:.2f}'
                cv2.putText(annotated_image, label_text, (int(prop.centroid[1]), int(prop.centroid[0])), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 139), 2)
    
    total_area = sum(hyphae_areas)
    average_area = np.mean(hyphae_areas) if hyphae_areas else 0
    average_aspect_ratio = np.mean(hyphae_aspect_ratios) if hyphae_aspect_ratios else 0

    fig, ax = plt.subplots(1, 2, figsize=(12, 6))
    ax[0].imshow(image, cmap='gray')
    ax[0].set_title('Original Image')
    ax[1].imshow(cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB))
    ax[1].set_title('Annotated Image')
    plt.show()

    
    print(f'Total Hyphae Cells: {hyphae_cells}')
    print(f'Total Hyphae Area: {total_area}')
    print(f'Average Hyphae Area: {average_area}')
    print(f'Average Hyphae Aspect Ratio: {average_aspect_ratio}')
    
    return {
        "Image Path": image_path,
        "Type": "Hyphae",
        "Total Cells": total_cells,
        "Total Specific Cells": hyphae_cells,
        "Total Area": total_area,
        "Average Area": average_area,
        "Average Aspect Ratio": average_aspect_ratio
    }


# Modify keys and results to include the new columns for the last three directories
def extract_last_three_dirs(image_path):
    parts = image_path.split(os.sep)
    if len(parts) >= 4:
        return parts[-4], parts[-3], parts[-2]
    else:
        return "N/A", "N/A", "N/A"


In [3]:

# directory = 'Biostat2L_21_microscopy/R02/10-22-24/11am/'
# image_files = glob.glob(os.path.join(directory, '*.jpg'))


In [4]:
# Grab all image files. 
directory = '../Biostat2L_22/Biostat22_microscope/'
image_files = collect_image_files(directory)

NameError: name 'os' is not defined

In [5]:
# length of image files. 
len(image_files)

NameError: name 'image_files' is not defined

## hyphae count #1 with darker images. 

In [6]:
threshold_value = 35 # changes the lighting. 
min_size = 50 # filters out small objects
pix = 20 #
min_area = 70 #
max_area = 250 #
ecc_threshold = 0.95
aspect_threshold = 4 #
solidity_threshold = 0.8 #
min_length = 15 #

results = []
for image_file in image_files:
    results.append(analyze_hyphae_cells(image_file,
                     threshold_value=threshold_value, 
                     min_size=min_size, 
                     pix=pix, 
                     min_area=min_area,
                     max_area=max_area,
                     ecc_threshold= ecc_threshold,
                     aspect_threshold=aspect_threshold, 
                     solidity_threshold=solidity_threshold, 
                     min_length=min_length))

NameError: name 'image_files' is not defined

In [7]:
csv_path = "hyphae_results.csv"
with open(csv_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    headers = list(results[0].keys())
    writer.writerow(headers)
    for result in results:
        writer.writerow(result.values())

NameError: name 'csv' is not defined

## hyphae count #2 with lighter images. 

In [8]:
threshold_value = 55 # changes the lighting. 
min_size = 50 # filters out small objects
pix = 20 #
min_area = 70 #
max_area = 250 #
ecc_threshold = 0.95
aspect_threshold = 4 #
solidity_threshold = 0.8 #
min_length = 15 #

results = []
for image_file in image_files:
    results.append(analyze_hyphae_cells(image_file,
                     threshold_value=threshold_value, 
                     min_size=min_size, 
                     pix=pix, 
                     min_area=min_area,
                     max_area=max_area,
                     ecc_threshold= ecc_threshold,
                     aspect_threshold=aspect_threshold, 
                     solidity_threshold=solidity_threshold, 
                     min_length=min_length))

NameError: name 'image_files' is not defined

### save #2 results. 

In [9]:

csv_path = "hyphae_results_late.csv"
with open(csv_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    headers = list(results[0].keys()) + ["Dir_3", "Dir_2", "Dir_1"]
    writer.writerow(headers)
    for result in results:
        path = result["Image Path"]
        dir_3, dir_2, dir_1 = extract_last_three_dirs(path)
        writer.writerow(list(result.values()) + [dir_3, dir_2, dir_1])


NameError: name 'csv' is not defined

## Read in and try to average files after manual combination. 

In [10]:
data = pd.read_excel('hyphae_results.xlsx', header = 1)

NameError: name 'pd' is not defined

In [11]:
data.columns

NameError: name 'data' is not defined

In [12]:
columns_to_average = [
    'Total Cells', 'Total Specific Cells', 'Total Area', 
    'Average Area', 'Average Aspect Ratio',    'Total Cells.1', 'Total Specific Cells.1', 'Total Area.1', 
    'Average Area.1', 'Average Aspect Ratio.1'
]

In [13]:
df_averaged = data.groupby(['Dir_3', 'Dir_2', 'Dir_1'])[columns_to_average].agg(['mean', 'std']).reset_index()


NameError: name 'data' is not defined

In [14]:
threshold_value = 50 # changes the lighting. 
min_size = 50 # filters out small objects
pix = 20 #
min_area = 70 #
max_area = 250 #
ecc_threshold = 0.95
aspect_threshold = 4 #
solidity_threshold = 0.7 #
min_length = 15 #

analyze_hyphae_cells(image_test,
                     threshold_value=threshold_value, 
                     min_size=min_size, 
                     pix=pix, 
                     min_area=min_area,
                     max_area=max_area,
                     ecc_threshold= ecc_threshold,
                     aspect_threshold=aspect_threshold, 
                     solidity_threshold=solidity_threshold, 
                     min_length=min_length)

NameError: name 'image_test' is not defined

## Yeast analysis. 

In [15]:


def analyze_yeast_cells(image_path, threshold_value, min_size=30, pix=255, min_area=10, max_area=200, ecc_threshold=0.95, aspect_threshold=4,solidity_threshold=0.5, min_length=30,max_length=200):
    '''
    function to process microscopy images to determine objects that are hyphae.
    the image processing puts the image through
    1. grayscale
    2. thresholding
    3. small object removal
    4. finds props in photo
    5. draws contours around the hyphae. 

    image_path - string to image file. 
    threshold_value - int of the threshold
    min_size - small object removal threshold
    pix - pixels to use in the processed image
    min_area - minimal area to count as hyphae
    max_area = max area to count as hyphae
    ecc_threshold - eccentric shape -- 0 = sphere, 1 = non-spherical
    aspect_threshold - len/width threshold.
    solidity_threshold
    min_length = min length to be hypahe cell.

    
    '''
    
    
    
    # Load the image
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError("Error: Image not found at given path.")
    # print(f"\n\n\n Analyzing {image_path} with params: {threshold_value}, {min_size}, {pix}, {min_area}, {max_area}")
    print(f"\nAnalyzing {image_path} with the following parameters:")
    print(f"  threshold_value: {threshold_value}")
    print(f"  min_size: {min_size}")
    print(f"  pix: {pix}")
    print(f"  min_area: {min_area}")
    print(f"  max_area: {max_area}")
    print(f"  ecc_threshold: {ecc_threshold}")
    print(f"  aspect_threshold: {aspect_threshold}")
    print(f"  solidity_threshold: {solidity_threshold}")
    print(f"  min_length: {min_length}")
    # Apply a threshold to segment the cells
    _, thresholded = cv2.threshold(image, threshold_value, pix, cv2.THRESH_BINARY_INV)
    
    # Remove small objects (noise)
    cleaned = morphology.remove_small_objects(thresholded.astype(bool), min_size=min_size).astype(np.uint8)

    # Label the connected components
    labels = measure.label(cleaned)
    props = measure.regionprops(labels)
    
    total_cells = 0
    yeast_cells = 0
    yeast_areas = []
    yeast_aspect_ratios = []
    
    # Create a copy of the original image to draw annotations
    annotated_image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)


    for prop in props:
        if min_area < prop.area:  # Threshold area to count as a cell, adjust as needed
            total_cells += 1

            mask = np.zeros_like(labels, dtype=np.uint8)
            mask[labels == prop.label] = 255

            contours, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_NONE)
            
            if len(contours) == 0:
                continue
            
            contour = contours[0]
            
            aspect_ratio = prop.major_axis_length / prop.minor_axis_length if prop.minor_axis_length != 0 else 0
            solidity = prop.area / prop.convex_area if prop.convex_area != 0 else 0
            major_length = prop.major_axis_length

            
            if (prop.eccentricity < ecc_threshold) and (solidity < solidity_threshold) and (prop.major_axis_length<max_length):
                yeast_cells += 1
                yeast_areas.append(prop.area)
                yeast_aspect_ratios.append(aspect_ratio)
                # cv2.drawContours(annotated_image, [contour], 0, (0, 0, 139), 2)
                cv2.drawContours(annotated_image, [contour], 0, (57, 255, 20), 2)
                
                label_text = f'Hyphae AR:{aspect_ratio:.2f}'
                cv2.putText(annotated_image, label_text, (int(prop.centroid[1]), int(prop.centroid[0])), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 139), 2)
    
    total_area = sum(yeast_areas)
    average_area = np.mean(yeast_areas) if yeast_areas else 0
    average_aspect_ratio = np.mean(yeast_aspect_ratios) if yeast_aspect_ratios else 0

    fig, ax = plt.subplots(1, 2, figsize=(20, 6))
    ax[0].imshow(image, cmap='gray')
    ax[0].set_title('Original Image')
    ax[1].imshow(cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB))
    ax[1].set_title('Annotated Image')
    plt.show()

    
    print(f'Total Hyphae Cells: {yeast_cells}')
    print(f'Total Hyphae Area: {total_area}')
    print(f'Average Hyphae Area: {average_area}')
    print(f'Average Hyphae Aspect Ratio: {average_aspect_ratio}')
    
    return {
        "Image Path": image_path,
        "Type": "Yeast",
        "Total Cells": total_cells,
        "Total Specific Cells": yeast_cells,
        "Total Area": total_area,
        "Average Area": average_area,
        "Average Aspect Ratio": average_aspect_ratio
    }




# Modify keys and results to include the new columns for the last three directories
def extract_last_three_dirs(image_path):
    parts = image_path.split(os.sep)
    if len(parts) >= 4:
        return parts[-4], parts[-3], parts[-2]
    else:
        return "N/A", "N/A", "N/A"


## Parameter search

In [16]:
def parameter_search(image_file, param_grid):
    keys = param_grid.keys()
    values = param_grid.values()
    
    results = []
    for combination in itertools.product(*values):
        params = dict(zip(keys, combination))
        result = analyze_yeast_cells(image_file, **params)
        results.append(result)
    
    return results

# Define the parameter grid
param_grid = {
    "threshold_value": [40,60,90],
    "min_size": [20],
    "pix": [20],
    "min_area": [10],
    "max_area": [500],
    "ecc_threshold": [0.7,0.8, 0.95],
    "aspect_threshold": [2, 4, 6],
    "solidity_threshold": [0.8, 0.7],
    "min_length": [10, 20]
}

# Example usage
# image_files = ['image1.jpg', 'image2.jpg', 'image3.jpg']  # Replace with actual image files
results = parameter_search(image_files[1], param_grid)


NameError: name 'image_files' is not defined

In [17]:
image_test = '../Biostat2L_22/Biostat22_microscope/R01/2024_1031/1245pm/Image_1778.jpg'

## yeast count #1. 

In [18]:
threshold_value = 50 # changes the lighting. 
min_size = 0 # filters out small objects
pix = 255 #
min_area = 10 #
max_area = 250 #
ecc_threshold = 0.95
aspect_threshold = 5 #
solidity_threshold = 0.75 #
min_length = 5 #
max_length = 50


results = []
for image_file in image_files:
    results.append(analyze_yeast_cells(image_file,
                     threshold_value=threshold_value, 
                     min_size=min_size, 
                     pix=pix, 
                     min_area=min_area,
                     max_area=max_area,
                     ecc_threshold= ecc_threshold,
                     aspect_threshold=aspect_threshold, 
                     solidity_threshold=solidity_threshold, 
                     min_length=min_length))

NameError: name 'image_files' is not defined

In [19]:
z

NameError: name 'z' is not defined

In [20]:
image_test = '../Biostat2L_22/Biostat22_microscope/R01/2024_1031/1245pm/Image_1778.jpg'

In [21]:
csv_path = "yeast_results.csv"
with open(csv_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    headers = list(results[0].keys())
    writer.writerow(headers)
    for result in results:
        writer.writerow(result.values())

NameError: name 'csv' is not defined

In [22]:
csv_path = "yeast_results_2.csv"

with open(csv_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    headers = list(results[0].keys()) + ["Dir_3", "Dir_2", "Dir_1"]
    writer.writerow(headers)
    for result in results:
        path = result["Image Path"]
        dir_3, dir_2, dir_1 = extract_last_three_dirs(path)
        writer.writerow(list(result.values()) + [dir_3, dir_2, dir_1])

NameError: name 'csv' is not defined

In [23]:


def analyze_yeast_cells(image_path, threshold_value, min_size=30, pix=255, min_area=10, max_area=200, ecc_threshold=0.95, aspect_threshold=4):
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError("Error: Image not found at given path.")
    print(f"\n\n\n Analyzing {image_path} with params: {threshold_value}, {min_size}, {pix}, {min_area}, {max_area}")

    _, thresholded = cv2.threshold(image, threshold_value, pix, cv2.THRESH_BINARY_INV)
    
    cleaned = morphology.remove_small_objects(thresholded.astype(bool), min_size=min_size).astype(np.uint8)

    labels = measure.label(cleaned)
    props = measure.regionprops(labels)
    
    total_cells = 0
    yeast_cells = 0
    yeast_areas = []
    yeast_aspect_ratios = []
    
    annotated_image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)

    for prop in props:
        if min_area < prop.area:
            total_cells += 1

            mask = np.zeros_like(labels, dtype=np.uint8)
            mask[labels == prop.label] = 255

            contours, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_NONE)
            
            if len(contours) == 0:
                continue
            
            contour = contours[0]
            
            aspect_ratio = prop.major_axis_length / prop.minor_axis_length if prop.minor_axis_length != 0 else 0
            
            if (prop.area < max_area) and (prop.area > min_area) and (aspect_ratio < aspect_threshold):
                yeast_cells += 1
                yeast_areas.append(prop.area)
                yeast_aspect_ratios.append(aspect_ratio)
                cv2.drawContours(annotated_image, [contour], 0, (0, 255, 0), 2)
                cv2.putText(annotated_image, 'Yeast', (int(prop.centroid[1]), int(prop.centroid[0])), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    
    total_area = sum(yeast_areas)
    average_area = np.mean(yeast_areas) if yeast_areas else 0
    average_aspect_ratio = np.mean(yeast_aspect_ratios) if yeast_aspect_ratios else 0

    print(f'Total Yeast Cells: {yeast_cells}')
    print(f'Total Yeast Area: {total_area}')
    print(f'Average Yeast Area: {average_area}')
    print(f'Average Yeast Aspect Ratio: {average_aspect_ratio}')
    
    return {
        "Image Path": image_path,
        "Type": "Yeast",
        "Total Cells": total_cells,
        "Total Specific Cells": yeast_cells,
        "Total Area": total_area,
        "Average Area": average_area,
        "Average Aspect Ratio": average_aspect_ratio
    }



In [24]:

csv_path = "cell_analysis_results.csv"
file_exists = os.path.isfile(csv_path)
with open(csv_path, mode='a', newline='') as file:
    writer = csv.writer(file)
    if not file_exists:
        writer.writerow(["Image Path", "Type", "Total Cells", "Total Specific Cells", "Total Area", "Average Area", "Average Aspect Ratio"])
    for result in results:
        writer.writerow(result.values())


NameError: name 'os' is not defined

In [25]:




def analyze_yeast_cells(image_path, threshold_value, min_size=30, pix=255, min_area=10, max_area=200, ecc_threshold=0.95, aspect_threshold=4):
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError("Error: Image not found at given path.")
    print(f"\n\n\n Analyzing {image_path} with params: {threshold_value}, {min_size}, {pix}, {min_area}, {max_area}")

    _, thresholded = cv2.threshold(image, threshold_value, pix, cv2.THRESH_BINARY_INV)
    
    cleaned = morphology.remove_small_objects(thresholded.astype(bool), min_size=min_size).astype(np.uint8)

    labels = measure.label(cleaned)
    props = measure.regionprops(labels)
    
    total_cells = 0
    yeast_cells = 0
    yeast_areas = []
    yeast_aspect_ratios = []
    
    annotated_image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)

    for prop in props:
        if min_area < prop.area:
            total_cells += 1

            mask = np.zeros_like(labels, dtype=np.uint8)
            mask[labels == prop.label] = 255

            contours, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_NONE)
            
            if len(contours) == 0:
                continue
            
            contour = contours[0]
            
            aspect_ratio = prop.major_axis_length / prop.minor_axis_length if prop.minor_axis_length != 0 else 0
            
            if (prop.area < max_area) and (prop.area > min_area) and (aspect_ratio < aspect_threshold):
                yeast_cells += 1
                yeast_areas.append(prop.area)
                yeast_aspect_ratios.append(aspect_ratio)
                cv2.drawContours(annotated_image, [contour], 0, (0, 255, 0), 2)
                cv2.putText(annotated_image, 'Yeast', (int(prop.centroid[1]), int(prop.centroid[0])), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    
    total_area = sum(yeast_areas)
    average_area = np.mean(yeast_areas) if yeast_areas else 0
    average_aspect_ratio = np.mean(yeast_aspect_ratios) if yeast_aspect_ratios else 0

    print(f'Total Yeast Cells: {yeast_cells}')
    print(f'Total Yeast Area: {total_area}')
    print(f'Average Yeast Area: {average_area}')
    print(f'Average Yeast Aspect Ratio: {average_aspect_ratio}')
    
    return {
        "Image Path": image_path,
        "Type": "Yeast",
        "Total Cells": total_cells,
        "Total Specific Cells": yeast_cells,
        "Total Area": total_area,
        "Average Area": average_area,
        "Average Aspect Ratio": average_aspect_ratio
    }



## yeast count 2.

In [26]:
threshold_value = 50 # changes the lighting. 
min_size = 0 # filters out small objects
pix = 255 #
min_area = 10 #
max_area = 250 #
ecc_threshold = 0.8
aspect_threshold = 4 #
solidity_threshold = 0.8 #
min_length = 5 #
max_length = 50


results = []
for image_file in image_files:
    results.append(analyze_yeast_cells(image_file,
                     threshold_value=threshold_value, 
                     min_size=min_size, 
                     pix=pix, 
                     min_area=min_area,
                     max_area=max_area,
                     ecc_threshold= ecc_threshold,
                     aspect_threshold=aspect_threshold, 
                     solidity_threshold=solidity_threshold, 
                     min_length=min_length))

NameError: name 'image_files' is not defined

In [27]:
results.append(analyze_yeast_cells(image_path, threshold_value=127))


NameError: name 'image_path' is not defined

In [28]:

csv_path = "cell_analysis_results.csv"
file_exists = os.path.isfile(csv_path)
with open(csv_path, mode='a', newline='') as file:
    writer = csv.writer(file)
    if not file_exists:
        writer.writerow(["Image Path", "Type", "Total Cells", "Total Specific Cells", "Total Area", "Average Area", "Average Aspect Ratio"])
    for result in results:
        writer.writerow(result.values())


NameError: name 'os' is not defined

## Read in and try to average files after manual combination. 

In [29]:
data = pd.read_excel('yeast_results_2.xlsx', header = 0)

NameError: name 'pd' is not defined

In [30]:
data.columns

NameError: name 'data' is not defined

In [31]:
columns_to_average = [
    'Total Cells', 'Total Specific Cells', 'Total Area', 
    'Average Area', 'Average Aspect Ratio',    
]

In [32]:
df_averaged = data.groupby(['Dir_3', 'Dir_2', 'Dir_1'])[columns_to_average].agg(['mean', 'std']).reset_index()


NameError: name 'data' is not defined

In [33]:
df_averaged.to_csv('yeast_averaged.csv')

NameError: name 'df_averaged' is not defined